# Batched calculation of auc roc

In [3]:
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from matplotlib import pyplot as plt

from IPython.display import display, Markdown

## 1. Motivation and framing the challenge

A main contribution of the project is being able to train an RL agent, such that it can learn the biasing 
dynamics of an acceptance loop and be therefore able to correct them.

To do so it needs to be able to compute some metrics on a subsample of the credit data at different stages
of the biasing process. Logically the longer a financial institution has existed and has performed applicants
selection, the larger the amount of data it will have.

For an off policy RL framework this means, that a replay buffer will need to handle features of shape
`[B, N, F]`, so batch size, sub sample of observations containing only "current information" - meaning
data from up to some round which has been observed, and some amount of features. Given that the data grows
with time, the amount of valid N entries will vary across batches. This makes that a normalized tensor of
shape `[B, N, F]` will *necessarily* contain unvalid positions.

Furthermore: for the RL framework to be feasibly trained it needs to be able to calculate the metric related
to the reward in a vectorized manner. Therefore the challenge is twofold:
 
1. Find a way to calculate in a vectorized manner `B` AUC-ROCs
2. Be able to find an adequate way to mask unvalid observations

## 2. Revisiting the calculation of the AUC-ROC

Firs revist quickly and rather loosely how the ROC-Curve for binary classification is defined. The ROC-Curve
is a rank based method to assess the discriminative ability of a binary classifier. Without loss of generality
we can frame the binary classification problem as a regression problem, which tries to predict if a statistical
unit with observable features $x \in \mathcal{X}$, with $\mathcal{X}$ some feature space containing any kind
of features, is part of a class $y \in \{0, 1\}$ (note that $0, 1$ is an arbitrary encoding). Thereby the 
underlying assumption is that an observation belong either of the two classes - so there is no third option.
Then we can define a classifier $c : \mathcal{X} \to \mathcal{S} \subseteq \mathbb{R}$ as a function mapping 
some covariates $x \in \mathcal{X}$ to a scoring $s \in \mathcal{S}$, such that $c(x) = s$. The higher the scoring
the more likely it should be that a statistical unit belongs to class $1$.

Given a classifier $c$, the ROC curve is a plot of the true positive rate (TPR) on the vertical axis against 
the false positive rate (FPR) on the horizontal axis for every possible threshold setting. If the available data
was infinite the ROC-Curve would therefore be impossible to calculate, but given a finite sample, many beautiful
simplifications can be and are done. Let our sample be of size $N \in \mathbb{N}$ be indexable by the set
$\mathbf{N} = \{1, \ldots, N\} \subset \mathbb{N}$. Given this index we can define the covariates, labels and
scores as sequences $X = (x_i)_{i \in \mathbf{N}}$, $Y = (y_i)_{i \in \mathbf{N}}$ and $S = (s_i)_{i \in \mathbf{N}}$
with $s_i \equiv c(x_i)$.

Now let's define $\mathbf{O} = (o_i)_{i \in \mathbf{N}}$ as a sequence ordering the observations by descending score.
Meaning $\forall i, j \in \mathbb{N}, i < j : s_{o_i} \leq s_{o_j}$. Then every possible (meaningful) threshold lies 
in $[s_{o_1}, s_{o_N}] \subset \mathbb{R}$. Of course $s_{o_1} = \min_{i \in \mathbf{N}} s_i$
and $s_{o_N} = \max_{i \in \mathbf{N}} s_i$. Furthermore, as we do not have a scoring for every value in this interval,
we can only calculate meaningful estimates for the TPR and FPR at the points where we have observed any data. Let's see an example
with synthetic data


In [11]:
N = 2048
repeated_scores = 128
proportion_y_equals_1 = 0.3

device = torch.device('cpu')
rng = torch.Generator(device=device).manual_seed(1807)

N_y_equals_1 = round(N * proportion_y_equals_1)
N_y_equals_0  = N - N_y_equals_1
true_prob = torch.cat([
    torch.rand(size=(N_y_equals_0,), generator=rng) * 0.4, # \in [0, 0.4)
    torch.rand(size=(N_y_equals_1,), generator=rng) * 0.4 + 0.6 # \in [0.6, 1)
])

labels = torch.bernoulli(true_prob, generator=rng)

desired_std_for_noise = 0.07 # Range of noise will be then around [-0.21,0.21]
score_noise = torch.randn(size=(N,), generator=rng) * desired_std_for_noise
scores = true_prob + score_noise

repeated_scores_y_1 = round(repeated_scores * 0.3)
repeated_scores_y_1 = repeated_scores_y_1 + (repeated_scores_y_1 % 2)
repeated_scores_y_0 = repeated_scores - repeated_scores_y_1

idx_to_repeat_y0 = torch.randperm(N_y_equals_0, generator=rng)[:repeated_scores_y_0].reshape(repeated_scores_y_0//2, 2)
idx_to_repeat_y1 = torch.randperm(N_y_equals_1, generator=rng)[:repeated_scores_y_1].reshape(repeated_scores_y_1//2, 2) + N_y_equals_0

idx_to_repeat = torch.cat([idx_to_repeat_y0, idx_to_repeat_y1], dim=0)

scores[idx_to_repeat[:, 0]] = scores[idx_to_repeat[:, 1]].clone() # make sure it is not a view

In [12]:
scores

tensor([0.1337, 0.2462, 0.2541,  ..., 0.7011, 0.7127, 0.5891])